# Turbo Image Studio
Use **ComfyUI on port 8188** to generate or edit images. Use this JupyterLab on **8888** to manage files, download models and open a terminal. Startup status is on **8080**.

Close ComfyUI's generic Templates gallery, open the Workflows sidebar and double-click one of the four bundled workflows. Upload an input image before running image-edit. The official Turbo checkpoint downloads lazily on the first queued request.

Open `MODEL_FOLDERS.md` for the complete folder map. The editable application is `ComfyUI/`, models are in `ComfyUI/models/`, saved workflows in `ComfyUI/user/default/workflows`, and outputs in `ComfyUI/output`. Original `turbo_*.png` files retain RGBA; SaveImage also produces RGB previews.

Files are durable only if `/workspace` is mounted on your own volume. Download outputs before stopping temporary storage. Volume storage has its own charge.


# Turbo workflow controls

Open Workflows and choose **Turbo-Generation**, **Turbo-Portrait**, **Turbo-Landscape** or **Turbo-Image-Edit**. All nodes and ComfyUI-Manager 4.2.2 are included. The older small workflows remain available for compatibility.

## Follow the connected graph

1. **Official Turbo Model / Precision** selects the pinned Qwen Image 2.1 Turbo checkpoint, BF16 and CPU offload. This node configures a pipeline handle; weights load on the first sampler execution.
2. **Qwen 2.1 Transformer LoRA** defaults to None. Upload a compatible transformer LoRA safetensors file to `/workspace/ComfyUI/models/loras`, refresh ComfyUI, select it and set strength. The loader and PEFT dependency are included. Old Qwen Image/Edit adapters are not automatically compatible with the different 2.1 architecture. Loading errors are reported; no weights are downloaded for you. Each request resets the previous adapter, so selecting None removes it.
3. **Prompt** contains positive and negative text. With the recommended CFG=1, negative text does not guide sampling. CFG above 1 requires a negative prompt and is experimental for Turbo.
4. **Sampling Settings** exposes the schedule, steps, CFG and KV cache. `official_8_steps`, steps=8, CFG=1 and KV=true preserve Qwen's recommendation. Setting another step count while retaining official schedule produces an explanatory error, never a silently ignored value. `experimental_linear` uses the chosen 1–50 steps with explicit call-time sigmas. `experimental_custom` requires a strictly descending JSON sigma list with exactly steps values in (0,1], excluding terminal zero. The scheduler remains FlowMatchEulerDiscreteScheduler and applies its configured shift. Other schedules/CFG values have not been evaluated by Qwen.
5. **Sampler** sets width, height and seed. An optional IMAGE connection enables single-image editing. The edit graph includes a LoadImage node and a bundled example image. Replace it by uploading/selecting your image. IMAGE output connects to ordinary image processing, preview and save nodes.
6. **Save / Preview** show and save results. Original RGBA outputs and JSON metadata are also written to `/workspace/outputs`.
7. **Actual Turbo Run Settings** displays model revision, actual steps, scheduler, saved/requested sigma inputs, CFG, prompts, KV state, precision, offload, adapter/strength, seed and timing after execution. This is the result of the run, not just a template description.

## Pipeline handles and native nodes

The graph exposes real configuration connections for the official Diffusers pipeline. `TURBO_MODEL`, `TURBO_CONDITIONING` and `TURBO_SAMPLING` are specific types, not native ComfyUI MODEL/CLIP/LATENT objects. Use the included Turbo LoRA loader and sampler with those handles. Native KSampler/LoRA Loader cannot replace them. Standard native model loaders remain installed for separate compatible workflows. This avoids claiming unsupported native checkpoint compatibility.

## Manager and dependencies

Manager is enabled through ComfyUI's supported `--enable-manager` integration. Open its extension/Manager control to browse installed nodes or the registry; standard security defaults are retained. All bundled workflows run without installing another node. Additional third-party node packages can change dependencies; the tested image remains the recovery baseline. Save user files and avoid updating the core blindly. Durable files require a volume mounted at `/workspace`.

## References

[Official Turbo sampling](https://huggingface.co/Qwen/Qwen-Image-2.1-Turbo#sampling), [official Diffusers pipeline](https://github.com/huggingface/diffusers/blob/85c9fa4caee33d1c5b473395784471d2d7957c1b/src/diffusers/pipelines/qwenimage21/pipeline_qwenimage21.py), [ComfyUI Manager setup](https://github.com/Comfy-Org/ComfyUI#comfyui-manager).


In [ ]:
from pathlib import Path
workspace = Path('/workspace')
for name in ('ComfyUI/models', 'ComfyUI/custom_nodes', 'ComfyUI/user/default/workflows', 'ComfyUI/input', 'ComfyUI/output', 'logs'):
    path = workspace / name
    print(name, 'exists:', path.exists())


## Add your own models
Use Upload for small files. For multi-GB checkpoints, choose **File → New → Terminal**, change to the correct model folder and download directly from the publisher. Check provenance, model format, filename and license first. Refresh ComfyUI after the download.

Checkpoints go in `ComfyUI/models/checkpoints`; diffusion models in `diffusion_models`; text encoders in `text_encoders`; VAEs in `vae`; compatible LoRAs in `loras`. A checkpoint or LoRA added here does **not** change the OfficialTurbo nodes, which use the pinned Qwen Diffusers model in `.cache/huggingface`. Use a compatible standard ComfyUI workflow for other models.

Custom nodes go in `ComfyUI/custom_nodes`. Install their requirements deliberately in a Terminal; startup does not execute third-party installation scripts or bundle ComfyUI-Manager. User-installed Python packages on container storage need reinstalling on a fresh container.


## If an interface will not open
A Pod marked Running may still be pulling the container. Wait for services on the 8080 status page to say ready. Open `logs/comfyui.log` and `logs/jupyter.log` here to diagnose errors. A ComfyUI startup failure leaves Jupyter running so you can inspect and fix files. Restart the Pod from RunPod after correcting the cause.

If Jupyter asks for a token, use your Pod's Jupyter connection/token under RunPod → Connect. Without one supplied by RunPod, use SSH and `jupyter server list` to retrieve the automatically generated token. The public template contains no shared password.

ComfyUI is unauthenticated by default for anyone with its proxy URL. Jupyter requires a per-Pod token. Do not publish either private connection credentials or private files.

Wrapper support: https://github.com/xerox7777777/runpod-turbo-image-studio/issues . Include version, hardware and relevant logs; remove credentials and private content.
